# SAMSum Dialogue Summarization using Qwen2.5-7B + LoRA

**Task 2 — Dialogue Summarization**

This Colab follows the same overall structure as the reference fine-tuning project, but changes the task and dataset to **SAMSum dialogue summarization**.

### Project configuration
- Dataset: `knkarthick/samsum`
- Task: Dialogue → concise summary
- Base model: `Qwen/Qwen2.5-7B-Instruct`
- Fine-tuning: LoRA / QLoRA-style 4-bit loading
- Training subset: 5,000 examples
- Training steps: 60
- Maximum sequence length: 2048
- LoRA rank: 16
- LoRA alpha: 16
- LoRA dropout: 0
- Random seed: 3407

The SAMSum dataset contains messenger-like conversations with human-written summaries.

## 1. Install dependencies
Run this first in a fresh Colab runtime with a GPU enabled.

In [1]:
!pip -q install -U unsloth transformers datasets trl accelerate bitsandbytes sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.7/75.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB

## 2. Check GPU

In [2]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")
else:
    print("WARNING: Select Runtime -> Change runtime type -> GPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


## 3. Load Qwen2.5-7B-Instruct in 4-bit

In [3]:
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print("Qwen2.5-7B-Instruct loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2.5-7B-Instruct loaded successfully.


## 4. Add the LoRA adapter

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
print("LoRA adapter added successfully.")

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapter added successfully.


## 5. Load the SAMSum dataset
The Hugging Face dataset uses the fields `dialogue`, `summary`, and `id`.

In [5]:
from datasets import load_dataset

dataset = load_dataset("knkarthick/samsum")
print(dataset)
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))
print(dataset["train"][0])

README.md:   0%|          | 0.00/4.36k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/9.26M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/504k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/522k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})
Train: 14731
Validation: 818
Test: 819
{'id': '13818513', 'dialogue': "Amanda: I baked  cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)", 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}


## 6. Select and shuffle 5,000 training examples

In [6]:
SEED = 3407
TRAIN_SAMPLES = 5000

train_dataset = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(min(TRAIN_SAMPLES, len(dataset["train"]))))
)
print("Selected training examples:", len(train_dataset))

Selected training examples: 5000


## 7. Format the data for supervised fine-tuning

In [7]:
def format_example(example):
    return {
        "text": (
            "Below is a conversation between multiple people.\n"
            "Provide a concise and informative summary in third person.\n\n"
            "### Conversation:\n"
            + example["dialogue"]
            + "\n\n### Summary:\n"
            + example["summary"]
        )
    }

train_text = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
print(train_text[0]["text"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Below is a conversation between multiple people.
Provide a concise and informative summary in third person.

### Conversation:
Sam: You there yet?
Sonia: Yeah. Just arrived.
Sam: What's the party like? Many people?
Sonia: Totally packed. Going to go and mingle with the rest.
Sam: OK. See you soon.

### Summary:
Sonia has just arrived at the party. There are many people. Sonia and Sam will see each other soon. 


## 8. Configure supervised fine-tuning
This is a short 60-step experiment matching the reference project's style.

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="samsum_qwen_lora_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
    save_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)
print("Trainer configured.")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

Trainer configured.


## 9. Train the LoRA adapter

In [9]:
import time
start_time = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start_time
print(f"Training finished in {elapsed:.2f} seconds ({elapsed/60:.2f} minutes).")
print(trainer_stats)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.695462
2,2.562249
3,2.575244
4,2.610830
5,2.199482
6,2.339289
7,2.091434
8,2.182115
9,1.997445
10,1.980872


Training finished in 470.89 seconds (7.85 minutes).
TrainOutput(global_step=60, training_loss=1.9491368114948273, metrics={'train_runtime': 468.3641, 'train_samples_per_second': 1.025, 'train_steps_per_second': 0.128, 'total_flos': 5090728804503552.0, 'train_loss': 1.9491368114948273, 'epoch': 0.096})


## 10. GPU memory statistics

In [10]:
if torch.cuda.is_available():
    print(f"Peak reserved GPU memory: {torch.cuda.max_memory_reserved()/1024**3:.3f} GB")
    print(f"Peak allocated GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.3f} GB")

Peak reserved GPU memory: 10.635 GB
Peak allocated GPU memory: 8.191 GB


## 11. Save the fine-tuned LoRA model

In [11]:
save_dir = "samsum_lora_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"LoRA model saved successfully to: {save_dir}/")

Unsloth: Restored added_tokens_decoder metadata in samsum_lora_model/tokenizer_config.json.


LoRA model saved successfully to: samsum_lora_model/


## 12. Inference on unseen conversations

In [12]:
FastLanguageModel.for_inference(model)

test_conversations = [
    "John: Are we still meeting at 3 PM?\nSarah: Yes, I'll be at the library.\nJohn: Great. I'll bring the documents.\nSarah: Please also bring the budget report.",
    "Mike: Did you finish the project report?\nTom: Almost. I still need to add the results section.\nMike: Okay, send it to me when you're done.\nTom: Sure, I'll send it tonight.",
    "Alice: Are you coming to the party tonight?\nBob: I don't think so. I have an exam tomorrow.\nAlice: No problem. Good luck with your exam!",
    "Emma: Can you pick up some groceries?\nDavid: Sure. What do we need?\nEmma: Milk, bread and eggs.\nDavid: I'll get them on my way home."
]

def summarize_dialogue(conversation):
    prompt = (
        "Below is a conversation between multiple people.\n"
        "Provide a concise and informative summary in third person.\n\n"
        "### Conversation:\n" + conversation +
        "\n\n### Summary:\n"
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

for i, conversation in enumerate(test_conversations, 1):
    print("=" * 80)
    print(f"FINE-TUNED MODEL TEST {i}")
    print("CONVERSATION:", conversation, sep="\n")
    print("\nMODEL OUTPUT:")
    print(summarize_dialogue(conversation))
    print()

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINE-TUNED MODEL TEST 1
CONVERSATION:
John: Are we still meeting at 3 PM?
Sarah: Yes, I'll be at the library.
John: Great. I'll bring the documents.
Sarah: Please also bring the budget report.

MODEL OUTPUT:


Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is a conversation between multiple people.
Provide a concise and informative summary in third person.

### Conversation:
John: Are we still meeting at 3 PM?
Sarah: Yes, I'll be at the library.
John: Great. I'll bring the documents.
Sarah: Please also bring the budget report.

### Summary:
John will meet Sarah at 3 PM at the library. John will bring the documents and the budget report.  Sarah will be there.  <sep>  John will meet Sarah at 3 PM. He will bring the documents and the budget report. Sarah will be at the library.  <sep>  John will meet Sarah at 3 PM. She will be at the library. John will bring the documents and the budget report.  <sep>  John and Sarah will meet at 3 PM. John will bring the documents and the budget report. Sarah will be at the library.  <sep>  John

FINE-TUNED MODEL TEST 2
CONVERSATION:
Mike: Did you finish the project report?
Tom: Almost. I still need to add the results section.
Mike: Okay, send it to me when you're done.
Tom: Sure, I'll send it tonigh

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is a conversation between multiple people.
Provide a concise and informative summary in third person.

### Conversation:
Mike: Did you finish the project report?
Tom: Almost. I still need to add the results section.
Mike: Okay, send it to me when you're done.
Tom: Sure, I'll send it tonight.

### Summary:
Tom will send Mike his project report tonight.  <sep> Tom is almost done with his project report.  <sep> Mike wants Tom to add the results section to his project report.  <sep> Tom will send Mike his project report tonight.  <sep> Mike asked Tom to send him his project report.  <sep> Tom will send Mike his project report tonight.  <sep> Tom will send Mike his project report tonight.  <sep> Mike needs Tom's project report.  <sep> Tom will send Mike his project report tonight.  <sep> Tom will send Mike his project report tonight

FINE-TUNED MODEL TEST 3
CONVERSATION:
Alice: Are you coming to the party tonight?
Bob: I don't think so. I have an exam tomorrow.
Alice: No problem. Good

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is a conversation between multiple people.
Provide a concise and informative summary in third person.

### Conversation:
Alice: Are you coming to the party tonight?
Bob: I don't think so. I have an exam tomorrow.
Alice: No problem. Good luck with your exam!

### Summary:
Bob won't come to the party tonight because he has an exam tomorrow. Alice wishes him good luck.  <sep>

FINE-TUNED MODEL TEST 4
CONVERSATION:
Emma: Can you pick up some groceries?
David: Sure. What do we need?
Emma: Milk, bread and eggs.
David: I'll get them on my way home.

MODEL OUTPUT:
Below is a conversation between multiple people.
Provide a concise and informative summary in third person.

### Conversation:
Emma: Can you pick up some groceries?
David: Sure. What do we need?
Emma: Milk, bread and eggs.
David: I'll get them on my way home.

### Summary:
David will buy milk, bread and eggs on his way home.  <sep> Emma asked David to buy milk, bread and eggs.  <sep> David will go shopping after work.  <sep> Em

## 13. Before vs After — Base Qwen2.5-7B-Instruct
The same test conversations are used for a qualitative comparison.

In [13]:
import gc
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

GPU memory cleared.


In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer # Import standard transformers

test_conversations = [
    "John: Are we still meeting at 3 PM?\nSarah: Yes, I'll be at the library.\nJohn: Great. I'll bring the documents.\nSarah: Please also bring the budget report.",
    "Mike: Did you finish the project report?\nTom: Almost. I still need to add the results section.\nMike: Okay, send it to me when you're done.\nTom: Sure, I'll send it tonight.",
    "Alice: Are you coming to the party tonight?\nBob: I don't think so. I have an exam tomorrow.\nAlice: No problem. Good luck with your exam!",
    "Emma: Can you pick up some groceries?\nDavid: Sure. What do we need?\nEmma: Milk, bread and eggs.\nDavid: I'll get them on my way home."
]

# Load base model using standard transformers to avoid unsloth's GPU-specific patching
base_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    load_in_4bit=True,
    device_map="cpu" # Force load on CPU
)
# FastLanguageModel.for_inference(base_model) # Removed, as it causes issues when model is on CPU

def base_summarize_dialogue(conversation):
    prompt = (
        "Below is a conversation between multiple people.\n"
        "Provide a concise and informative summary in third person.\n\n"
        "### Conversation:\n" + conversation +
        "\n\n### Summary:\n"
    )
    # Move inputs to CPU since the model is now on CPU
    inputs = base_tokenizer([prompt], return_tensors="pt").to("cpu")
    outputs = base_model.generate(**inputs, max_new_tokens=128, use_cache=True)
    return base_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

for i, conversation in enumerate(test_conversations, 1):
    print("=" * 80)
    print(f"BASE MODEL TEST {i}")
    print("CONVERSATION:", conversation, sep="\n")
    print("\nBASE MODEL OUTPUT:")
    print(base_summarize_dialogue(conversation))
    print()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

## 14. Qualitative comparison

In [1]:
comparison_notes = {
    "Evaluation type": "Qualitative",
    "Training subset": "5,000 SAMSum examples",
    "Training steps": 60,
    "Task": "Dialogue summarization",
    "Base model": "Qwen2.5-7B-Instruct",
    "Fine-tuning": "LoRA with 4-bit loading",
}
for k, v in comparison_notes.items():
    print(f"{k}: {v}")

Evaluation type: Qualitative
Training subset: 5,000 SAMSum examples
Training steps: 60
Task: Dialogue summarization
Base model: Qwen2.5-7B-Instruct
Fine-tuning: LoRA with 4-bit loading


## 15. Download the LoRA adapter
Run after training to create a ZIP containing the adapter and tokenizer.

In [2]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("samsum_qwen2.5_7b_lora_model", "zip", "samsum_lora_model")
print("Created:", zip_path)
files.download(zip_path)

Created: /content/samsum_qwen2.5_7b_lora_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 16. Project Documentation Summary

**Title:** Dialogue Summarization using Qwen2.5-7B-Instruct and LoRA Fine-Tuning

**Dataset:** SAMSum (`knkarthick/samsum`)

**Objective:** Adapt Qwen2.5-7B-Instruct to generate concise summaries of multi-speaker conversations.

**Workflow:** SAMSum → 5,000 examples → formatting → 4-bit Qwen2.5-7B-Instruct → LoRA → 60-step training → unseen dialogue inference → base-vs-fine-tuned comparison → saved LoRA adapter.

**Limitation:** The 60-step run is an initial experiment. It should not be presented as proof that the fine-tuned model is objectively better without a larger evaluation and quantitative metrics such as ROUGE.